# Imports

In [7]:
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

import os
import re

# Setup

In [8]:
# Set root directory for project
current_file_dir = os.getcwd() # Get current directory
ROOT_DIR = os.path.dirname(current_file_dir) # Get project root

# Helper Functions

In [9]:
def show_and_save_md(df: pd.DataFrame, name: str):
    """
    Cleans mileage strings and converts km/kg to kmpl.
    
    Args:
        df (pd.DataFrame): The dataframe to be used
        name (string): The name of the md file
    """
    df_md = df.to_markdown() # Convert to markdown

    with open(f"{ROOT_DIR}/reports/{name}.md", "w", encoding="utf-8") as f:
        f.write(df_md) # Save markdown file to reports

    print(df_md) # Show markdown table

# Ingestion

In [10]:
df = pd.read_csv(f'{ROOT_DIR}/data_raw/train.csv') # Load in data

In [11]:
df.shape # Check shape of data

(5847, 14)

In [12]:
df.head() # Show first few rows of data

,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


# Cleaning

## Remove Units & Unit Conversions

### Mileage

In [13]:
df['Mileage'].head(10)

0    19.67 kmpl
1      13 km/kg
2    20.77 kmpl
3     15.2 kmpl
4    23.08 kmpl
5    11.36 kmpl
6    20.54 kmpl
7     22.3 kmpl
8    21.56 kmpl
9     16.8 kmpl
Name: Mileage, dtype: str

In [14]:
def clean_mileage(val: str | float) -> float:
    """
    Cleans mileage strings and converts km/kg to kmpl.
    
    Args:
        val (str or float): The raw mileage string

    Returns:
        float: The converted mileage in kmpl. If the input was not a 
            string or did not match known units, returns the original value.
    """
    
    if not isinstance(val, str):
        return val
    
    if val.endswith('kmpl'):
        val = re.sub(f' kmpl', '', val)
        return float(val)
    elif val.endswith('km/kg'):
        val = re.sub(f' km/kg', '', val)
        val = float(val)
        return round(val / 0.74, 2)
    return val

In [15]:
df['Mileage'] = df['Mileage'].apply(clean_mileage)

In [16]:
df['Mileage'].head(10)

0    19.67
1    17.57
2    20.77
3    15.20
4    23.08
5    11.36
6    20.54
7    22.30
8    21.56
9    16.80
Name: Mileage, dtype: float64

### Engien

In [17]:
df['Engine'].head(10)

0    1582 CC
1    1199 CC
2    1248 CC
3    1968 CC
4    1461 CC
5    2755 CC
6    1598 CC
7    1248 CC
8    1462 CC
9    1497 CC
Name: Engine, dtype: str

In [18]:
def clean_engine(val: str | float) -> float:
    """
    Cleans engine strings and removes CC.
    
    Args:
        val (str or float): The raw engine string.

    Returns:
        float: Cleaned engine string.
    """
    
    if not isinstance(val, str):
        return val
    
    if val.endswith('CC'):
        val = re.sub(f' CC', '', val)
        return float(val)
    return val

In [19]:
df['Engine'] = df['Engine'].apply(clean_engine)

In [20]:
df['Engine'].head(10)

0    1582.0
1    1199.0
2    1248.0
3    1968.0
4    1461.0
5    2755.0
6    1598.0
7    1248.0
8    1462.0
9    1497.0
Name: Engine, dtype: float64

### Power

In [21]:
df['Power'].head(10)

0     126.2 bhp
1      88.7 bhp
2     88.76 bhp
3     140.8 bhp
4      63.1 bhp
5     171.5 bhp
6     103.6 bhp
7        74 bhp
8    103.25 bhp
9     116.3 bhp
Name: Power, dtype: str

In [22]:
def clean_power(val: str | float) -> float:
    """
    Cleans power strings and removes bhp.
    
    Args:
        val (str or float): The raw power string.

    Returns:
        float: Cleaned power string.
    """
    
    if not isinstance(val, str):
        return val
    
    if val.endswith('bhp'):
        val = re.sub(f' bhp', '', val)
        return float(val)
    return val

In [23]:
df['Power'] = df['Power'].apply(clean_power)

In [24]:
df['Power'].head(10)

0    126.20
1     88.70
2     88.76
3    140.80
4     63.10
5    171.50
6    103.60
7     74.00
8    103.25
9    116.30
Name: Power, dtype: float64

### New_Price

In [25]:
df['New_Price'].head(10)

0           NaN
1     8.61 Lakh
2           NaN
3           NaN
4           NaN
5       21 Lakh
6           NaN
7           NaN
8    10.65 Lakh
9           NaN
Name: New_Price, dtype: str

In [26]:
def clean_new_price(val: str | float) -> float:
    """
    Cleans New_Price strings and removes Lakh.
    
    Args:
        val (str or float): The raw New_Price string.

    Returns:
        float: Cleaned New_Price string.
    """
    
    if not isinstance(val, str):
        return val
    
    if val.endswith('Lakh'):
        val = re.sub(f' Lakh', '', val)
        return float(val)
    return val

In [27]:
df['New_Price'] = df['New_Price'].apply(clean_new_price)

In [28]:
df['New_Price'].head(10)

0      NaN
1     8.61
2      NaN
3      NaN
4      NaN
5     21.0
6      NaN
7      NaN
8    10.65
9      NaN
Name: New_Price, dtype: object

## Handle Missing Data

In [29]:
missing_values_percents = df.isna().mean() * 100 # Get missing values percents

show_and_save_md(missing_values_percents, "missing_values_percents")

|                   |          0 |
|:------------------|-----------:|
| Unnamed: 0        |  0         |
| Name              |  0         |
| Location          |  0         |
| Year              |  0         |
| Kilometers_Driven |  0         |
| Fuel_Type         |  0         |
| Transmission      |  0         |
| Owner_Type        |  0         |
| Mileage           |  0.0342056 |
| Engine            |  0.6157    |
| Power             |  0.6157    |
| Seats             |  0.649906  |
| New_Price         | 86.0612    |
| Price             |  0         |


In [30]:
# Drop New_Price
df.drop("New_Price", axis=1, inplace=True)

***Justification for dropping***

The column ```New_Price``` was missing 86% of the values. Imputing these values is not a good option because we don't have anough data to understand what values should be imputed. It also violates the 30% rule saying that columns with more than 30% missing values should be dropped.

In [31]:
numeric_cols = df.select_dtypes(include='number').columns # Get numeric cols
numeric_imputer = SimpleImputer(strategy='median') # Init imputer
df[numeric_cols] = numeric_imputer.fit_transform(df[numeric_cols]) # Impute

***Justification for median impute***

These columns are numerical data with a small amount missing. Imputing the median is an ideal way to fill the missing values without scewing the distribution.

In [32]:
categorical_cols = df.select_dtypes(include='str').columns # Get categorical cols
categorical_imputer = SimpleImputer(strategy='most_frequent') # Init imputer
df[categorical_cols] = categorical_imputer.fit_transform(df[categorical_cols]) # Impute

***Justification for mode frequent impute***

These columns are categorical data with a small amount missing. Imputing the most frequent category will fill in the missing data without scewing the outcomes too much.

In [33]:
df.isna().mean() * 100 # Showing all missing values filled or dropped

Unnamed: 0           0.0
Name                 0.0
Location             0.0
Year                 0.0
Kilometers_Driven    0.0
Fuel_Type            0.0
Transmission         0.0
Owner_Type           0.0
Mileage              0.0
Engine               0.0
Power                0.0
Seats                0.0
Price                0.0
dtype: float64

## Encoding

In [34]:
encoder = OneHotEncoder(sparse_output=False) # Init 
encoding_columns = ["Fuel_Type", "Transmission"] # Set cols to be encoded

encoded_data = encoder.fit_transform(df[encoding_columns]) # One hot encode cols
encoded_df = pd.DataFrame( # Convert to df
    encoded_data,
    columns=encoder.get_feature_names_out(encoding_columns),
    index=df.index
)
df = pd.concat([df, encoded_df], axis=1) # Add encoded cols to original df

df.head() # Check result

,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual
0,1.0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015.0,41000.0,Diesel,Manual,First,19.67,1582.0,126.20,5.0,12.50,1.0,0.0,0.0,0.0,1.0
1,2.0,Honda Jazz V,Chennai,2011.0,46000.0,Petrol,Manual,First,17.57,1199.0,88.70,5.0,4.50,0.0,0.0,1.0,0.0,1.0
2,3.0,Maruti Ertiga VDI,Chennai,2012.0,87000.0,Diesel,Manual,First,20.77,1248.0,88.76,7.0,6.00,1.0,0.0,0.0,0.0,1.0
3,4.0,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013.0,40670.0,Diesel,Automatic,Second,15.20,1968.0,140.80,5.0,17.74,1.0,0.0,0.0,1.0,0.0
4,6.0,Nissan Micra Diesel XV,Jaipur,2013.0,86999.0,Diesel,Manual,First,23.08,1461.0,63.10,5.0,3.50,1.0,0.0,0.0,0.0,1.0


## Create New Feature

***New Feature: ```Mileage_To_Engine_Ratio```***

Tells us how efficient the engine is.

In [35]:
df["Mileage_To_Engine_Ratio"] = df["Mileage"] / df["Engine"] # Create new feature

In [36]:
df[["Mileage", "Engine", "Mileage_To_Engine_Ratio"]].head() # Check

,Mileage,Engine,Mileage_To_Engine_Ratio
0,19.67,1582.0,0.012434
1,17.57,1199.0,0.014654
2,20.77,1248.0,0.016643
3,15.20,1968.0,0.007724
4,23.08,1461.0,0.015797


# Analysis

## Select

See most efficient car fuel type with selected cols.

In [37]:
# Same as done above but adding groupby Fuel_Type
df_selected = (
    df[["Mileage", "Engine", "Fuel_Type", "Mileage_To_Engine_Ratio"]]   # select
    .groupby("Fuel_Type")
    .mean(numeric_only=True)
    .reset_index()
)

show_and_save_md(df_selected, "SELECT_most_efficient_car")

|    | Fuel_Type   |   Mileage |   Engine |   Mileage_To_Engine_Ratio |
|---:|:------------|----------:|---------:|--------------------------:|
|  0 | Diesel      |   18.6555 |  1865.08 |                 0.0116245 |
|  1 | Electric    |   18.19   |   935    |                 0.131378  |
|  2 | Petrol      |   17.5782 |  1355.23 |                 0.0146685 |


## Filter

Filter by type of car to buy for customer and then look for cheapest location to buy.

In [38]:
df_filter = df[ # Filter
    (df['Mileage'] < 100_000) & 
    (df['Transmission'] == 'Automatic') & 
    (df['Fuel_Type'] == 'Petrol')
    ].groupby("Location")['Price'].mean()

show_and_save_md(df_filter, "FILTER_low_miles_automatic_petrol_cheapest_location")

| Location   |    Price |
|:-----------|---------:|
| Ahmedabad  |  7.13923 |
| Bangalore  | 15.205   |
| Chennai    |  6.58703 |
| Coimbatore | 13.1414  |
| Delhi      | 14.2024  |
| Hyderabad  | 10.4443  |
| Jaipur     |  4.65882 |
| Kochi      | 13.989   |
| Kolkata    | 14.9786  |
| Mumbai     |  9.8537  |
| Pune       |  7.98885 |


## Rename

Rename ```Price``` to ```Sale_Price```

In [39]:
list(df.columns)

['Unnamed: 0',
 'Name',
 'Location',
 'Year',
 'Kilometers_Driven',
 'Fuel_Type',
 'Transmission',
 'Owner_Type',
 'Mileage',
 'Engine',
 'Power',
 'Seats',
 'Price',
 'Fuel_Type_Diesel',
 'Fuel_Type_Electric',
 'Fuel_Type_Petrol',
 'Transmission_Automatic',
 'Transmission_Manual',
 'Mileage_To_Engine_Ratio']

In [40]:
df.rename(columns={"Price": "Sale_Price"}, inplace=True)

In [41]:
list(df.columns)

['Unnamed: 0',
 'Name',
 'Location',
 'Year',
 'Kilometers_Driven',
 'Fuel_Type',
 'Transmission',
 'Owner_Type',
 'Mileage',
 'Engine',
 'Power',
 'Seats',
 'Sale_Price',
 'Fuel_Type_Diesel',
 'Fuel_Type_Electric',
 'Fuel_Type_Petrol',
 'Transmission_Automatic',
 'Transmission_Manual',
 'Mileage_To_Engine_Ratio']

## Mutate

Mutating ```Year``` to not have decinal by changing it to int

In [42]:
df.head()

,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Sale_Price,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual,Mileage_To_Engine_Ratio
0,1.0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015.0,41000.0,Diesel,Manual,First,19.67,1582.0,126.20,5.0,12.50,1.0,0.0,0.0,0.0,1.0,0.012434
1,2.0,Honda Jazz V,Chennai,2011.0,46000.0,Petrol,Manual,First,17.57,1199.0,88.70,5.0,4.50,0.0,0.0,1.0,0.0,1.0,0.014654
2,3.0,Maruti Ertiga VDI,Chennai,2012.0,87000.0,Diesel,Manual,First,20.77,1248.0,88.76,7.0,6.00,1.0,0.0,0.0,0.0,1.0,0.016643
3,4.0,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013.0,40670.0,Diesel,Automatic,Second,15.20,1968.0,140.80,5.0,17.74,1.0,0.0,0.0,1.0,0.0,0.007724
4,6.0,Nissan Micra Diesel XV,Jaipur,2013.0,86999.0,Diesel,Manual,First,23.08,1461.0,63.10,5.0,3.50,1.0,0.0,0.0,0.0,1.0,0.015797


In [43]:
df["Year"] = df["Year"].astype(int) # Mutate Year to remove decimal

In [44]:
df.head()

,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Sale_Price,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual,Mileage_To_Engine_Ratio
0,1.0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000.0,Diesel,Manual,First,19.67,1582.0,126.20,5.0,12.50,1.0,0.0,0.0,0.0,1.0,0.012434
1,2.0,Honda Jazz V,Chennai,2011,46000.0,Petrol,Manual,First,17.57,1199.0,88.70,5.0,4.50,0.0,0.0,1.0,0.0,1.0,0.014654
2,3.0,Maruti Ertiga VDI,Chennai,2012,87000.0,Diesel,Manual,First,20.77,1248.0,88.76,7.0,6.00,1.0,0.0,0.0,0.0,1.0,0.016643
3,4.0,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670.0,Diesel,Automatic,Second,15.20,1968.0,140.80,5.0,17.74,1.0,0.0,0.0,1.0,0.0,0.007724
4,6.0,Nissan Micra Diesel XV,Jaipur,2013,86999.0,Diesel,Manual,First,23.08,1461.0,63.10,5.0,3.50,1.0,0.0,0.0,0.0,1.0,0.015797


## Arrange

Arrange ```Year``` in descending order

In [45]:
df_arrange = df.sort_values(by="Year")
df_arrange

,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Sale_Price,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual,Mileage_To_Engine_Ratio
5558,5716.0,Maruti Zen LX,Jaipur,1998,95150.0,Petrol,Manual,Third,17.30,993.0,60.00,5.0,0.53,0.0,0.0,1.0,0.0,1.0,0.017422
3039,3138.0,Maruti Zen LXI,Jaipur,1998,95150.0,Petrol,Manual,Third,17.30,993.0,60.00,5.0,0.45,0.0,0.0,1.0,0.0,1.0,0.017422
3630,3749.0,Mercedes-Benz E-Class 250 D W 210,Mumbai,1998,55300.0,Diesel,Automatic,First,10.00,1796.0,157.70,5.0,3.90,1.0,0.0,0.0,1.0,0.0,0.005568
1791,1845.0,Honda City 1.3 EXI,Pune,1999,140000.0,Petrol,Manual,First,13.00,1343.0,90.00,5.0,0.90,0.0,0.0,1.0,0.0,1.0,0.009680
1185,1224.0,Maruti Zen VX,Jaipur,1999,70000.0,Petrol,Manual,Second,17.30,993.0,60.00,5.0,0.77,0.0,0.0,1.0,0.0,1.0,0.017422
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4956,5102.0,Maruti Wagon R VXI,Kochi,2019,31817.0,Petrol,Manual,First,22.50,998.0,67.04,5.0,5.34,0.0,0.0,1.0,0.0,1.0,0.022545
5712,5875.0,Mercedes-Benz C-Class Progressive C 220d,Ahmedabad,2019,4000.0,Diesel,Automatic,First,0.00,1950.0,194.00,5.0,35.00,1.0,0.0,0.0,1.0,0.0,0.000000
1486,1534.0,Honda City i VTEC VX Option,Coimbatore,2019,23882.0,Petrol,Manual,First,17.40,1497.0,117.30,5.0,11.87,0.0,0.0,1.0,0.0,1.0,0.011623
4319,4452.0,Toyota Innova Crysta 2.8 GX AT,Kochi,2019,14076.0,Diesel,Automatic,First,11.36,2755.0,171.50,7.0,19.40,1.0,0.0,0.0,1.0,0.0,0.004123


## Summarize

I want to see which location has the oldest cars for sale

In [46]:
df_summary = (
    df[["Location", "Mileage", "Year"]]   # select
    .groupby("Location")
    .mean(numeric_only=True)
    .sort_values("Year")
    .reset_index()
)

show_and_save_md(df_summary, "SUMMARIZE_oldest_car_by_location")

|    | Location   |   Mileage |    Year |
|---:|:-----------|----------:|--------:|
|  0 | Chennai    |   18.2462 | 2012.07 |
|  1 | Pune       |   17.8866 | 2012.42 |
|  2 | Jaipur     |   19.1898 | 2012.61 |
|  3 | Hyderabad  |   18.6773 | 2012.83 |
|  4 | Bangalore  |   16.7405 | 2012.84 |
|  5 | Kolkata    |   19.2017 | 2013.13 |
|  6 | Ahmedabad  |   18.861  | 2013.34 |
|  7 | Delhi      |   17.7244 | 2013.35 |
|  8 | Mumbai     |   17.2481 | 2013.35 |
|  9 | Coimbatore |   17.8442 | 2015.42 |
| 10 | Kochi      |   18.5855 | 2015.53 |


# Save

In [47]:
df.to_csv(f"{ROOT_DIR}/data_clean/train_clean.csv", index=False) # Save cleaned & processed df